# cTreeBalls Ly-alpha: 1D versus 3D

This notebook compares the native radial 2PCF/3PCF estimators with the 3D octree estimators. It uses an exactly collinear catalog, projects the **3D numerator and denominator** onto the 1D bins, and only then normalizes. This is the geometry in which the estimators should agree exactly. For a real non-collinear survey, the 1D method intentionally ignores transverse distance and is not the same statistic as the 3D method.

In [ ]:
from pathlib import Path
import sys

def find_repository():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        script = candidate / 'examples' / 'compare_lya_1d_3d.py'
        if script.is_file():
            return candidate
    raise RuntimeError('Open this notebook from the cTreeBalls source tree')

repo = find_repository()
sys.path.insert(0, str(repo / 'examples'))

from compare_lya_1d_3d import ComparisonConfig, run_comparison

repo

## Run every Ly-alpha profile

This runs the 3D 2PCF and 3PCF, the radial scan versions, the radial 2PCF interval tree, and both combined methods. Reduce `n_pixels` while exploring 3PCF settings; its direct triplet work grows much faster than the 2PCF work.

In [ ]:
config = ComparisonConfig(
    output_dir=repo / 'examples' / 'output_lya_1d_vs_3d_notebook',
    cballs_executable=repo / 'cballs',
    n_pixels=160,
    pixels_per_forest=4,
    threads=4,
    rp_max=12.0,
    rp_bins=12,
    rt_max=2.0,
    rt_bins=4,
    r3_max=8.0,
    r3_bins=8,
    theta_bins=4,
    mu_bins=4,
)

summary = run_comparison(config)

In [ ]:
print('Numerical comparisons')
for name, metrics in summary['numerical_comparisons'].items():
    print(f"{name:42s} max relative={metrics['max_relative']:.3e}  "
          f"max absolute={metrics['max_absolute']:.3e}")

print('\nWall times')
for method, seconds in summary['timings_seconds'].items():
    print(f'{method:30s} {seconds:.6f} s')

In [ ]:
from IPython.display import Image, display
display(Image(filename=summary['plot']))

## Inspect machine-readable results

The output directory contains `comparison_2pcf.csv`, `comparison_3pcf.csv`, `timings.csv`, `summary.json`, the generated catalog, and each method's full cTreeBalls output and log.

In [ ]:
import numpy as np

table_2pcf = np.genfromtxt(
    config.output_dir / 'comparison_2pcf.csv', delimiter=',', names=True
)
table_2pcf[['bin', 'xi_3d_projected', 'xi_1d_scan', 'xi_1d_tree',
             'relative_scan_vs_3d', 'relative_tree_vs_3d']]

## Use an existing catalog

Set `catalog_path` to a six-column `x y z delta weight forest_id` file. The comparator validates that all positions lie on the same observer-centered ray. A general survey catalog should instead be analyzed separately with the 1D and 3D estimators, because no exact projection comparison exists.

In [ ]:
# existing_config = ComparisonConfig(
#     output_dir=repo / 'examples' / 'output_existing_collinear_catalog',
#     cballs_executable=repo / 'cballs',
#     catalog_path=Path('/absolute/path/to/collinear_catalog.txt'),
#     threads=4,
# )
# existing_summary = run_comparison(existing_config)